In [1]:
import warnings
from pathlib import Path

import numpy as np
import polars as pl
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, pipeline
import os

warnings.filterwarnings('ignore')

CWD = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
DATA_DIR = CWD / 'data' / 'Stock_news'
MODEL_DIR = CWD / 'models'
MODEL_DIR.mkdir(exist_ok=True)

NEWS_PATH = DATA_DIR / 'subset_news.parquet'
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')

/opt/anaconda3/envs/ie_env/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Device: mps


In [2]:
# Toggle: True = 1k-row in-memory test (never touches full caches).
SAMPLE = False
N_SAMPLE = 1000
WRITE_CACHE = not SAMPLE

# Full-data cache paths (read-only when SAMPLE=True).
SUMMARIES_PATH  = DATA_DIR / 'lsa_summaries.parquet'
ARTICLE_EMB_PATH = DATA_DIR / 'article_embeddings.npy'
ARTICLE_IDS_PATH = DATA_DIR / 'article_embedding_ids.parquet'

# Model output (suffixed in sample mode so we don't clobber a full run).
suffix = '_sample' if SAMPLE else ''
TOPIC_MODEL_DIR = MODEL_DIR / f'bertopic_zeroshot{suffix}'

print('SAMPLE:', SAMPLE, '| WRITE_CACHE:', WRITE_CACHE)
print('Topic model dir:', TOPIC_MODEL_DIR)

SAMPLE: False | WRITE_CACHE: True
Topic model dir: /Users/armandhubler/Documents/coding_project/nlp_project/models/bertopic_zeroshot


In [3]:
POOL = [
    'AAPL','MSFT','GOOG','GOOGL','AMZN','TSLA','META','NVDA','AMD','INTC','CRM','NFLX',
    'ADBE','PYPL','UBER','SQ','SHOP','ZM','SNAP','COIN','PLTR','ORCL',
    'QQQ','SPY','DIA','IWM','T','VZ',
    'JPM','GS','MS','WFC','BAC','C',
    'XOM','CVX',
    'JNJ','PFE','MRNA','GILD','MRK','UNH','ABT',
    'WMT','COST','TGT','HD','KO','PEP','SBUX','MCD',
    'BA','GE','CAT','MMM','DIS','CMCSA','V','MA',
    'MU','QCOM','TXN','AVGO','F','GM',
]

df = pl.read_parquet(NEWS_PATH).filter(pl.col('Stock_symbol').is_in(POOL))
print(f'Full filtered df: {df.shape[0]:,} rows')

if SAMPLE:
    work = df.sample(n=min(N_SAMPLE, df.shape[0]), seed=42)
else:
    work = df
print(f'Working set:     {work.shape[0]:,} rows')
work.head(2)

Full filtered df: 139,522 rows
Working set:     139,522 rows


id,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary,date_parsed
u32,str,str,str,str,str,str,str,str,str,str,str,date
0,"""2019-12-09 00:00:00 UTC""","""Notable Monday Option Activity…","""GS""","""https://www.nasdaq.com/article…",null,null,"""Among the underlying component…","""Particularly high volume was s…","""Particularly high volume was s…","""Below is a chart showing GS's …","""Particularly high volume was s…",2019-12-09
1,"""2019-05-01 00:00:00 UTC""","""Facebook Stock Enters Its Seco…","""GOOG""","""https://www.nasdaq.com/article…",null,null,"""I had long believed Facebook (…","""While Amazon, Google and Micro…","""Like Microsoft (NASDAQ:), Appl…","""Like Microsoft (NASDAQ:), Appl…","""Like Microsoft (NASDAQ:), Appl…",2019-05-01


Push to R2 bucket

In [6]:
import boto3

# R2 credentials
ACCOUNT_ID = ""
ACCESS_KEY = ""
SECRET_KEY = ""
BUCKET_NAME = "semantic-beat"

juris = ''


# Create S3 client for R2
s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name="auto"
)

# # Upload parquet file
# file_path = "/Users/armandhubler/Documents/coding_project/nlp_project/data/Stock_news/subset_news.parquet"
# object_name = "subset_news.parquet"

# s3.upload_file(file_path, BUCKET_NAME, object_name)

print("Uploaded successfully 🚀")

Uploaded successfully 🚀


In [8]:
import os
import io
import boto3
import torch
import polars as pl
from tqdm import tqdm
from pathlib import Path
from transformers import AutoTokenizer, AutoModel, pipeline

BATCH_SIZE = 32
OBJECT_NAME = "finbert_embeddings.parquet"


# --- Helpers ---
def r2_file_exists(bucket, key):
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except Exception:
        return False

def read_parquet_from_r2(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pl.read_parquet(io.BytesIO(obj["Body"].read()))

def write_parquet_to_r2(df, bucket, key):
    buf = io.BytesIO()
    df.write_parquet(buf)
    buf.seek(0)
    s3.upload_fileobj(buf, bucket, key)

# Load model
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
embed_model = AutoModel.from_pretrained("ProsusAI/finbert")
sentiment_pipe = pipeline("text-classification", model="ProsusAI/finbert")
embed_model.eval()

# Save models locally
tokenizer.save_pretrained(MODEL_DIR / "finbert_tokenizer")
embed_model.save_pretrained(MODEL_DIR / "finbert_model")
sentiment_pipe.save_pretrained(MODEL_DIR / "finbert_sentiment")

# --- Check already processed ---
if r2_file_exists(BUCKET_NAME, OBJECT_NAME):
    existing_df = read_parquet_from_r2(BUCKET_NAME, OBJECT_NAME)
    processed_ids = set(existing_df["id"].to_list())
else:
    existing_df = None
    processed_ids = set()

df_todo = df.filter(~pl.col("id").is_in(processed_ids))
print(f"Total: {len(df)} | Processed: {len(processed_ids)} | Remaining: {len(df_todo)}")

# --- Inference loop ---
ids_list = df_todo["id"].to_list()
texts_list = df_todo["Lsa_summary"].to_list()

for start in tqdm(
    range(0, len(df_todo), BATCH_SIZE),
    desc="Processing batches",
    unit="batch",
    total=(len(df_todo) + BATCH_SIZE - 1) // BATCH_SIZE
):
    batch_ids = ids_list[start:start + BATCH_SIZE]
    batch_texts = texts_list[start:start + BATCH_SIZE]

    embeddings, sentiments, scores = [], [], []

    for text in batch_texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True)
        with torch.no_grad():
            outputs = embed_model(**inputs)

        embeddings.append(outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy())

        result = sentiment_pipe(text[:512])[0]
        sentiments.append(result["label"])
        scores.append(result["score"])

    emb_cols = {f"emb_{i}": [e[i] for e in embeddings] for i in range(768)}
    batch_df = pl.DataFrame({
        "id": batch_ids,
        "sentiment": sentiments,
        "score": scores,
        **emb_cols
    })

    if existing_df is not None:
        existing_df = pl.concat([existing_df, batch_df])
    else:
        existing_df = batch_df

    write_parquet_to_r2(existing_df, BUCKET_NAME, OBJECT_NAME)

KeyboardInterrupt: 

In [5]:
# OUTPUT_PATH = os.path.join(DATA_DIR, "finbert_embeddings.parquet")
# BATCH_SIZE = 32

# # Load model
# tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
# embed_model = AutoModel.from_pretrained("ProsusAI/finbert")
# sentiment_pipe = pipeline("text-classification", model="ProsusAI/finbert")
# embed_model.eval()

# # --- Check already processed ---
# if Path(OUTPUT_PATH).exists():
#     processed_ids = set(pl.read_parquet(OUTPUT_PATH, columns=['id'])['id'].to_list())
# else:
#     processed_ids = set()

# df_todo = df.filter(~pl.col('id').is_in(processed_ids))
# print(f"Total: {len(df)} | Processed: {len(processed_ids)} | Remaining: {len(df_todo)}")

# # --- Inference loop ---
# ids_list = df_todo['id'].to_list()
# texts_list = df_todo['Lsa_summary'].to_list()

# for start in tqdm(range(0, len(df_todo), BATCH_SIZE), desc="Processing batches", unit="batch", total=(len(df_todo) + BATCH_SIZE - 1) // BATCH_SIZE):
#     batch_ids = ids_list[start:start + BATCH_SIZE]
#     batch_texts = texts_list[start:start + BATCH_SIZE]

#     embeddings, sentiments, scores = [], [], []

#     for text in batch_texts:
#         inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True)
#         with torch.no_grad():
#             outputs = embed_model(**inputs)
#         embeddings.append(outputs.last_hidden_state[:, 0, :].squeeze().numpy())

#         result = sentiment_pipe(text[:512])[0]
#         sentiments.append(result["label"])
#         scores.append(result["score"])

#     emb_cols = {f"emb_{i}": [e[i] for e in embeddings] for i in range(768)}
#     batch_df = pl.DataFrame({'id': batch_ids, "sentiment": sentiments, "score": scores, **emb_cols})

#     if Path(OUTPUT_PATH).exists():
#         pl.concat([pl.read_parquet(OUTPUT_PATH), batch_df]).write_parquet(OUTPUT_PATH)
#     else:
#         batch_df.write_parquet(OUTPUT_PATH)

Device set to use mps:0


Total: 139522 | Processed: 23680 | Remaining: 115842


Processing batches:   0%|          | 0/3621 [00:00<?, ?batch/s]

KeyboardInterrupt: 